# Protein Disorder Prediction Analysis
This script processes IUPred2A disorder prediction output and FASTA sequence data to visualize intrinsically disordered regions in proteins.

In [ ]:
import pandas as pd

# CELL 1: Load and inspect the IUPred2A output file
# IUPred2A is a tool that predicts intrinsically disordered regions in proteins
# The output file contains per-residue disorder and anchor scores

# Define the path to the IUPred2A results file
file_path = 'C:/Users/IUPred2A_output-2025/NCCC_intra_protein_id.result'

# Open and read the file line by line
with open(file_path, 'r') as file:
    # Read all lines into a list to examine the file structure
    file_contents = file.readlines()

# Display the first 10 lines to understand the file format
file_contents[:10]  # Display the first 10 lines for inspection


In [ ]:
# CELL 2: Parse the IUPred2A output file into a structured DataFrame
# This cell extracts protein name, position, amino acid, and prediction scores

# Initialize an empty list to store parsed data rows
data = []

# Variable to store the protein name as we parse the file
protein_name = None

# Parse each line in the file
for line in file_contents:
    # Lines starting with '>' contain the protein identifier/name
    if line.startswith('>'):
        # Extract protein name by removing the '>' character
        protein_name = line.strip()[1:]
    
    # Skip comment lines (starting with '#') and empty lines
    elif line.startswith('#') or not line.strip():
        continue
    
    # Process data lines containing: position, amino_acid, iupred_score, anchor_score
    else:
        # Split the line into its constituent columns
        pos, amino_acid, iupred_score, anchor_score = line.strip().split()
        
        # Add this residue's data to our list
        # - pos: position in the protein sequence (converted to integer)
        # - amino_acid: single letter amino acid code
        # - iupred_score: disorder prediction score (0-1, higher = more disordered)
        # - anchor_score: anchor score (0-1, higher = anchoring residue)
        # - protein_name: which protein this residue belongs to
        data.append([int(pos), amino_acid, float(iupred_score), float(anchor_score), protein_name])

# Create a pandas DataFrame from the parsed data
# This provides easy filtering, grouping, and data manipulation
df = pd.DataFrame(data, columns=["POS", "AMINO_ACID", "IUPRED_SCORE", "ANCHOR_SCORE", "Pro"])

# Display the first few rows to verify the parsing was successful
df.head()


In [ ]:
# CELL 3: Load and inspect the FASTA file
# The FASTA file contains specific protein sequences and their coordinates
# This is used to identify and highlight regions of interest in the plots

# Define the path to the FASTA file containing the protein sequences of interest
fasta_file_path = "C:/Users/Syrine/Desktop/these2022/WorkSpaceThesis2024/IDP/NCCC_intra_pos.fasta"

# Open and read the FASTA file
with open(fasta_file_path, 'r') as file:
    # Read all lines into a list
    fasta_contents = file.readlines()

# Display the first few lines to understand the FASTA file structure
# FASTA files alternate between header lines (starting with '>') and sequence lines
fasta_contents[:10]


In [12]:
# CELL 4: Parse the FASTA file to extract protein names and sequence positions
# This extracts the coordinates where each sequence of interest is located in the full protein

# Initialize an empty list to store parsed FASTA data
fasta_data = []

# Parse the FASTA file (pairs of header and sequence lines)
# Step through the file in increments of 2 (one header, one sequence)
for i in range(0, len(fasta_contents), 2):
    
    # Extract and clean the header line (remove whitespace)
    header = fasta_contents[i].strip()
    
    # Extract and clean the sequence line
    sequence = fasta_contents[i + 1].strip()
    
    # Store the full header for position extraction
    pos = header
    
    # Extract the starting position from the header
    # The last underscore-separated value is the start position
    start_position = pos.split('_')[-1]
    
    # Remove the '>' character from the beginning of the header
    parts = header.lstrip('>')
    
    # Remove the last component (which is the start_position)
    # This leaves us with the protein identifier components
    parts = ''.join(parts.rsplit('_', 1)[0])
    
    # Split the protein identifier into components
    parts = parts.split('_')
    
    # Keep the first component and skip components at index 1 and 2 (likely indices)
    # This reconstruction gives us the protein name
    filtered_parts = parts[:1] + parts[3:]

    # Reconstruct the protein name by joining the filtered parts back together
    parts = '_'.join(filtered_parts)
    
    # Store the cleaned protein name
    protein_name = parts
    
    # Calculate the end position of this sequence fragment
    # End = Start + length of the sequence
    end_position = int(start_position) + len(sequence)
    
    # Add this protein's region information to our list
    fasta_data.append([protein_name, start_position, end_position])

# Create a DataFrame from the FASTA data
# This makes it easy to match proteins between the IUPred results and FASTA files
fasta_df = pd.DataFrame(fasta_data, columns=["Pro", "Start", "End"])

# Display the first few rows to verify the parsing
fasta_df.head()

# Convert the 'Pro' column to string type (in case any proteins are stored as numbers)
fasta_df['Pro'] = fasta_df['Pro'].astype(str)

# Display the modified DataFrame
fasta_df.head()

# Save the parsed FASTA data to a CSV file for reference/debugging
fasta_df.to_csv("fasta.csv")


# BAR PLOTS AND VISUALIZATIONS
Generate line plots showing IUPRED and ANCHOR scores across the protein sequence, with highlighted regions from the FASTA file.

In [13]:
import matplotlib.pyplot as plt
import os
import re

# CELL 5: Generate visualization plots for each protein
# This creates line plots showing disorder (IUPRED) and anchor predictions

# Configure matplotlib for consistent, publication-quality output
plt.rcParams.update({'font.size': 16})  # Set default font size to 16pt
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial']  # Use Arial font for consistency
})

# Create the output directory for plots if it doesn't already exist
output_dir = "bar_plots"
os.makedirs(output_dir, exist_ok=True)

# Generate a plot for each protein in the FASTA DataFrame
for _, row in fasta_df.iterrows():
    # Extract protein information from the current row
    protein_name = row['Pro']
    start_pos = row['Start']  # Starting position of the region of interest
    end_pos = row['End']      # Ending position of the region of interest
    
    # Filter the IUPred data to only include this specific protein
    protein_data = df[df['Pro'] == protein_name]
    
    # Create a new figure with specified size (10 inches wide, 6 inches tall)
    plt.figure(figsize=(10, 6))
    
    # Plot IUPRED_SCORE (disorder prediction) as a blue line
    # Higher scores indicate more disordered regions
    plt.plot(protein_data["POS"], protein_data["IUPRED_SCORE"], label="IUPRED_SCORE", color="blue")
    
    # Plot ANCHOR_SCORE (anchoring potential) as a green line
    # Higher scores indicate residues that anchor to interaction partners
    plt.plot(protein_data["POS"], protein_data["ANCHOR_SCORE"], label="ANCHOR_SCORE", color="green")

    # Add a horizontal red dashed line at y=0.5
    # This is typically the threshold for disorder prediction
    # (scores above 0.5 are considered disordered)
    plt.axhline(y=0.5, color='red', linestyle='--')
    
    # Add an orange background shading to highlight the region of interest
    # This visually emphasizes the area defined in the FASTA file
    plt.axvspan(int(start_pos), int(end_pos), color="orange", alpha=0.3, label=f"Highlight (POS {start_pos} to {end_pos})")
    
    # Add axis labels
    plt.xlabel("POS")           # X-axis: position in protein sequence
    plt.ylabel("Score")         # Y-axis: prediction score (0-1)
    
    # Add informative title showing which protein is being plotted
    plt.title(f"IUPRED vs ANCHOR Score for {protein_name}")
    
    # Add a legend to identify the different lines and shaded region
    plt.legend()
    
    # Enable grid lines for easier value reading
    plt.grid(True)
    
    # Adjust layout to prevent label cutoff
    plt.tight_layout()
    
    # Configure high-resolution output
    dpi_value = 1000  # 1000 dots per inch for publication-quality output
    width_in_inches = 1000 / dpi_value   # Calculate width in inches
    height_in_inches = width_in_inches   # Keep 1:1 aspect ratio
    
    # Create a safe filename by replacing any special characters with underscores
    # This prevents file system errors with invalid characters
    safe_protein_name = re.sub(r'[\/:*?"<>|]', '_', protein_name)
    
    # Save the plot as a PNG file in the output directory
    # Using the protein name in the filename for easy identification
    plt.savefig(os.path.join(output_dir, f"{safe_protein_name}.png"), dpi=dpi_value, bbox_inches='tight')
    
    # Close the plot to free up memory and prevent accumulation
    plt.close()  # This prevents the plot from being displayed in each iteration
